
# RIASEC Model Training — CatBoost

This notebook performs the complete modeling stage for the **RIASEC Explorer** project.

It starts from the preprocessed dataset produced by `riasec_preprocessing.ipynb` and:

1. Verifies the mapped dataset
2. Converts the 15 academic categories into the final 7 domains
3. Removes `general / interdisciplinary`
4. Builds the 54 model features
5. Imputes the remaining missing RIASEC item responses
6. Creates stratified train / validation / test splits
7. Computes square-root balanced class weights
8. Trains the final CatBoost multiclass classifier
9. Evaluates validation and untouched test performance
10. Saves the trained model and metadata files

The untouched test set is used only for the final evaluation.


## 1. Install dependencies

In [1]:
!pip install -q catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 5.9 MB/s eta 0:00:00


## 2. Imports and configuration

In [2]:
import json
import pickle
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    top_k_accuracy_score
)

from catboost import CatBoostClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")

Libraries imported successfully.



## 3. Load the preprocessed dataset

The notebook loads the dataset directly from the GitHub repository, so it can be run from a fresh Colab session with **Run all**.

Expected input: `data/final_dataset_mapped_only.csv`


In [3]:
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "nourElHouda-18/orienta_eureka_team/"
    "main/data/final_dataset_mapped_only.csv"
)

LOCAL_FALLBACK = Path("/content/final_dataset_mapped_only.csv")

try:
    df = pd.read_csv(DATA_URL, low_memory=False)
    print("Dataset loaded from GitHub.")
except Exception as e:
    print("GitHub loading failed:", e)
    if LOCAL_FALLBACK.exists():
        df = pd.read_csv(LOCAL_FALLBACK, low_memory=False)
        print("Dataset loaded from /content/final_dataset_mapped_only.csv")
    else:
        raise FileNotFoundError(
            "Could not load the dataset from GitHub and no local fallback was found. "
            "Upload final_dataset_mapped_only.csv to /content/ and run again."
        )

print("Shape:", df.shape)

Dataset loaded from GitHub.
Shape: (73484, 72)


## 4. Verify the mapped dataset

In [4]:
print("=" * 70)
print("DATASET VERIFICATION")
print("=" * 70)

print("Rows:", len(df))
print("Columns:", df.shape[1])
print("Missing major_category:", df["major_category"].isna().sum())
print("Number of major categories:", df["major_category"].nunique())

print("\nMajor-category distribution:")
print(df["major_category"].value_counts())

assert df.shape == (73484, 72), f"Unexpected dataset shape: {df.shape}"
assert df["major_category"].isna().sum() == 0, "Missing major_category values found."
assert df["major_category"].nunique() == 15, "Expected 15 major categories."

print("\nDataset verification passed.")

DATASET VERIFICATION
Rows: 73484
Columns: 72
Missing major_category: 0
Number of major categories: 15

Major-category distribution:
major_category
social & behavioral sciences        17131
business, economics & management    13076
health & welfare                     8762
languages & humanities               6834
engineering & built environment      6260
natural sciences & mathematics       5838
education                            2973
arts & design                        2918
computing & it                       2583
law & criminal justice               2433
communication & media                2198
general / interdisciplinary          1053
hospitality & services                629
public & international affairs        472
agriculture & animal sciences         324
Name: count, dtype: int64

Dataset verification passed.



## 5. Map the 15 categories to the final 7 domains

`general / interdisciplinary` is intentionally removed because it does not represent a specific vocational domain.


In [5]:
DOMAIN_MAP = {
    "arts & design": "Arts & Design",

    "business, economics & management": "Business & Management",
    "hospitality & services": "Business & Management",

    "education": "Education",

    "health & welfare": "Health",

    "languages & humanities": "Humanities & Communication",
    "communication & media": "Humanities & Communication",

    "engineering & built environment": "STEM",
    "natural sciences & mathematics": "STEM",
    "computing & it": "STEM",
    "agriculture & animal sciences": "STEM",

    "social & behavioral sciences": "Social & Public Sciences",
    "law & criminal justice": "Social & Public Sciences",
    "public & international affairs": "Social & Public Sciences"
}

model_df = df[
    df["major_category"] != "general / interdisciplinary"
].copy()

model_df["domain"] = model_df["major_category"].map(DOMAIN_MAP)

expected_counts = {
    "Social & Public Sciences": 20036,
    "STEM": 15005,
    "Business & Management": 13705,
    "Humanities & Communication": 9032,
    "Health": 8762,
    "Education": 2973,
    "Arts & Design": 2918
}

actual_counts = model_df["domain"].value_counts().to_dict()

print("Rows after removing General / Interdisciplinary:", len(model_df))
print("\nFinal domain distribution:")
print(model_df["domain"].value_counts())

assert len(model_df) == 72431, f"Expected 72,431 rows, found {len(model_df)}"
assert model_df["domain"].isna().sum() == 0, "Unmapped final domains found."
assert actual_counts == expected_counts, "Final domain counts differ from the expected dataset."

print("\n7-domain mapping verified successfully.")

Rows after removing General / Interdisciplinary: 72431

Final domain distribution:
domain
Social & Public Sciences      20036
STEM                          15005
Business & Management         13705
Humanities & Communication     9032
Health                         8762
Education                      2973
Arts & Design                  2918
Name: count, dtype: int64

7-domain mapping verified successfully.


## 6. Define the 54 model features

In [6]:
riasec_cols = (
    [f"R{i}" for i in range(1, 9)] +
    [f"I{i}" for i in range(1, 9)] +
    [f"A{i}" for i in range(1, 9)] +
    [f"S{i}" for i in range(1, 9)] +
    [f"E{i}" for i in range(1, 9)] +
    [f"C{i}" for i in range(1, 9)]
)

score_cols = [
    "R_score",
    "I_score",
    "A_score",
    "S_score",
    "E_score",
    "C_score"
]

feature_cols = riasec_cols + score_cols

missing_columns = [c for c in feature_cols if c not in model_df.columns]
assert not missing_columns, f"Missing feature columns: {missing_columns}"

print("RIASEC items:", len(riasec_cols))
print("RIASEC scores:", len(score_cols))
print("Total features:", len(feature_cols))

assert len(feature_cols) == 54

RIASEC items: 48
RIASEC scores: 6
Total features: 54



## 7. Verify and impute remaining missing RIASEC responses

A small number of item-level responses are missing because the preprocessing stage allowed respondents with at least 6 of 8 answered items in every RIASEC dimension.

Each missing item is imputed with that respondent's mean score for the corresponding RIASEC dimension.


In [7]:
missing_raw_before = int(model_df[riasec_cols].isna().sum().sum())
missing_scores = int(model_df[score_cols].isna().sum().sum())

print("Missing raw RIASEC responses before imputation:", missing_raw_before)
print("Missing calculated RIASEC scores:", missing_scores)

assert missing_raw_before == 6388, (
    f"Expected 6,388 missing raw responses, found {missing_raw_before}"
)
assert missing_scores == 0, "Calculated RIASEC scores should not contain missing values."

for letter in ["R", "I", "A", "S", "E", "C"]:
    dimension_cols = [f"{letter}{i}" for i in range(1, 9)]
    score_col = f"{letter}_score"

    for col in dimension_cols:
        model_df[col] = model_df[col].fillna(model_df[score_col])

missing_after = int(model_df[feature_cols].isna().sum().sum())

print("Missing model-feature values after imputation:", missing_after)

assert missing_after == 0, "Missing feature values remain after imputation."

print("Imputation verified.")

Missing raw RIASEC responses before imputation: 6388
Missing calculated RIASEC scores: 0
Missing model-feature values after imputation: 0
Imputation verified.


## 8. Prepare features and labels

In [8]:
X = model_df[feature_cols].copy()
y_text = model_df["domain"].copy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_text)

print("Final classes:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{i} -> {label}")

assert len(label_encoder.classes_) == 7
assert X.shape == (72431, 54)

print("\nX shape:", X.shape)
print("y shape:", y.shape)

Final classes:
0 -> Arts & Design
1 -> Business & Management
2 -> Education
3 -> Health
4 -> Humanities & Communication
5 -> STEM
6 -> Social & Public Sciences

X shape: (72431, 54)
y shape: (72431,)



## 9. Stratified train / validation / test split

- Train: approximately 64%
- Validation: approximately 16%
- Test: 20%

The test set remains untouched until the final evaluation.


In [9]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

assert X_train.shape == (46355, 54)
assert X_val.shape == (11589, 54)
assert X_test.shape == (14487, 54)

print("\nSplit verified.")

Train: (46355, 54)
Validation: (11589, 54)
Test: (14487, 54)

Split verified.


## 10. Compute square-root balanced class weights

In [10]:
classes = np.unique(y_train)

balanced_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

sqrt_weights = np.sqrt(balanced_weights)

print("Square-root balanced class weights:")
for cls, weight in zip(classes, sqrt_weights):
    label = label_encoder.inverse_transform([cls])[0]
    print(f"{label:30s} {weight:.4f}")

Square-root balanced class weights:
Arts & Design                  1.8833
Business & Management          0.8689
Education                      1.8654
Health                         1.0867
Humanities & Communication     1.0704
STEM                           0.8304
Social & Public Sciences       0.7186


## 11. Train the final CatBoost model

In [11]:
catboost_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=7,
    loss_function="MultiClass",
    class_weights=sqrt_weights.tolist(),
    random_seed=RANDOM_STATE,
    allow_writing_files=False,
    verbose=100
)

catboost_model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100,
    use_best_model=True
)

print("\nBest iteration:", catboost_model.get_best_iteration())

0:	learn: 1.9212465	test: 1.9222696	best: 1.9222696 (0)	total: 560ms	remaining: 9m 19s
100:	learn: 1.4934202	test: 1.5596055	best: 1.5596055 (100)	total: 58.3s	remaining: 8m 39s
200:	learn: 1.4094675	test: 1.5335614	best: 1.5335614 (200)	total: 1m 39s	remaining: 6m 35s
300:	learn: 1.3458553	test: 1.5220142	best: 1.5220142 (300)	total: 2m 21s	remaining: 5m 29s
400:	learn: 1.2905363	test: 1.5180920	best: 1.5180920 (400)	total: 3m 2s	remaining: 4m 33s
500:	learn: 1.2416867	test: 1.5167537	best: 1.5167187 (497)	total: 3m 42s	remaining: 3m 42s
600:	learn: 1.1941673	test: 1.5164555	best: 1.5160158 (528)	total: 4m 23s	remaining: 2m 55s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.516015816
bestIteration = 528

Shrink model to first 529 iterations.

Best iteration: 528


## 12. Evaluation function

In [12]:
def evaluate_model(model, X_eval, y_eval, name):
    probabilities = model.predict_proba(X_eval)
    predictions = np.argmax(probabilities, axis=1)

    results = {
        "accuracy": accuracy_score(y_eval, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_eval, predictions),
        "macro_precision": precision_score(
            y_eval, predictions, average="macro", zero_division=0
        ),
        "macro_recall": recall_score(
            y_eval, predictions, average="macro", zero_division=0
        ),
        "macro_f1": f1_score(
            y_eval, predictions, average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            y_eval, predictions, average="weighted", zero_division=0
        ),
        "top2_accuracy": top_k_accuracy_score(
            y_eval,
            probabilities,
            k=2,
            labels=np.arange(len(label_encoder.classes_))
        ),
        "top3_accuracy": top_k_accuracy_score(
            y_eval,
            probabilities,
            k=3,
            labels=np.arange(len(label_encoder.classes_))
        )
    }

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    print(f"Accuracy:           {results['accuracy']:.4f}")
    print(f"Balanced Accuracy:  {results['balanced_accuracy']:.4f}")
    print(f"Macro Precision:    {results['macro_precision']:.4f}")
    print(f"Macro Recall:       {results['macro_recall']:.4f}")
    print(f"Macro F1:           {results['macro_f1']:.4f}")
    print(f"Weighted F1:        {results['weighted_f1']:.4f}")
    print(f"Top-2 Accuracy:     {results['top2_accuracy']:.4f}")
    print(f"Top-3 Accuracy:     {results['top3_accuracy']:.4f}")

    return results, predictions, probabilities

## 13. Validation-set evaluation

In [13]:
validation_results, val_predictions, val_probabilities = evaluate_model(
    catboost_model,
    X_val,
    y_val,
    "CATBOOST — VALIDATION SET"
)


CATBOOST — VALIDATION SET
Accuracy:           0.4529
Balanced Accuracy:  0.3890
Macro Precision:    0.3944
Macro Recall:       0.3890
Macro F1:           0.3885
Weighted F1:        0.4484
Top-2 Accuracy:     0.6762
Top-3 Accuracy:     0.8045



Reference results from the selected experiment were approximately:

- Accuracy: 0.4529
- Balanced Accuracy: 0.3890
- Macro F1: 0.3885
- Top-2 Accuracy: 0.6762
- Top-3 Accuracy: 0.8045

Small differences can occur across CatBoost/library versions.


## 14. Final untouched test-set evaluation

In [14]:
test_results, test_predictions, test_probabilities = evaluate_model(
    catboost_model,
    X_test,
    y_test,
    "CATBOOST — FINAL TEST SET"
)

print("\nClassification report:\n")
print(
    classification_report(
        y_test,
        test_predictions,
        target_names=label_encoder.classes_,
        digits=4,
        zero_division=0
    )
)


CATBOOST — FINAL TEST SET
Accuracy:           0.4465
Balanced Accuracy:  0.3862
Macro Precision:    0.3929
Macro Recall:       0.3862
Macro F1:           0.3871
Weighted F1:        0.4430
Top-2 Accuracy:     0.6731
Top-3 Accuracy:     0.8071

Classification report:

                            precision    recall  f1-score   support

             Arts & Design     0.2701    0.3168    0.2916       584
     Business & Management     0.4746    0.5217    0.4970      2741
                 Education     0.2642    0.1714    0.2080       595
                    Health     0.4313    0.3744    0.4009      1752
Humanities & Communication     0.3302    0.2955    0.3119      1807
                      STEM     0.4888    0.5358    0.5112      3001
  Social & Public Sciences     0.4913    0.4874    0.4894      4007

                  accuracy                         0.4465     14487
                 macro avg     0.3929    0.3862    0.3871     14487
              weighted avg     0.4420    0.4465   


Reference results from the selected experiment were approximately:

- Accuracy: **0.4465**
- Balanced Accuracy: **0.3862**
- Macro Precision: **0.3929**
- Macro Recall: **0.3862**
- Macro F1: **0.3871**
- Weighted F1: **0.4430**
- Top-2 Accuracy: **0.6731**
- Top-3 Accuracy: **0.8071**

The application presents the Top-3 domains because RIASEC interests can overlap across multiple vocational domains.


## 15. Save the trained model and metadata

In [15]:
OUTPUT_DIR = Path("/content/riasec_model_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = OUTPUT_DIR / "riasec_catboost_final.cbm"
ENCODER_PATH = OUTPUT_DIR / "riasec_label_encoder.pkl"
FEATURES_PATH = OUTPUT_DIR / "riasec_features.json"
DOMAINS_PATH = OUTPUT_DIR / "riasec_domains.json"
METRICS_PATH = OUTPUT_DIR / "riasec_metrics.json"

catboost_model.save_model(str(MODEL_PATH))

with open(ENCODER_PATH, "wb") as f:
    pickle.dump(label_encoder, f)

with open(FEATURES_PATH, "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2)

with open(DOMAINS_PATH, "w", encoding="utf-8") as f:
    json.dump(label_encoder.classes_.tolist(), f, indent=2)

metrics_to_save = {
    "validation": {k: float(v) for k, v in validation_results.items()},
    "test": {k: float(v) for k, v in test_results.items()},
    "best_iteration": int(catboost_model.get_best_iteration()),
    "random_state": RANDOM_STATE,
    "n_features": len(feature_cols),
    "n_classes": len(label_encoder.classes_),
    "train_rows": len(X_train),
    "validation_rows": len(X_val),
    "test_rows": len(X_test)
}

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics_to_save, f, indent=2)

print("Saved model package files:")
for p in [MODEL_PATH, ENCODER_PATH, FEATURES_PATH, DOMAINS_PATH, METRICS_PATH]:
    print("-", p)

Saved model package files:
- /content/riasec_model_output/riasec_catboost_final.cbm
- /content/riasec_model_output/riasec_label_encoder.pkl
- /content/riasec_model_output/riasec_features.json
- /content/riasec_model_output/riasec_domains.json
- /content/riasec_model_output/riasec_metrics.json


## 16. Create a ZIP package

In [16]:
ZIP_PATH = Path("/content/riasec_final_model_files.zip")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for file_path in OUTPUT_DIR.iterdir():
        if file_path.is_file():
            z.write(file_path, arcname=file_path.name)

print("Model package created:")
print(ZIP_PATH)

Model package created:
/content/riasec_final_model_files.zip


## 17. Final run summary

In [17]:
print("=" * 70)
print("FINAL MODELING SUMMARY")
print("=" * 70)

print("Input mapped rows:             ", len(df))
print("Rows after 7-domain mapping:   ", len(model_df))
print("Number of features:            ", len(feature_cols))
print("Number of domains:             ", len(label_encoder.classes_))
print("Training rows:                 ", len(X_train))
print("Validation rows:               ", len(X_val))
print("Test rows:                     ", len(X_test))
print("Best CatBoost iteration:       ", catboost_model.get_best_iteration())

print("\nFinal test metrics:")
print(f"Accuracy:                       {test_results['accuracy']:.4f}")
print(f"Balanced Accuracy:              {test_results['balanced_accuracy']:.4f}")
print(f"Macro F1:                       {test_results['macro_f1']:.4f}")
print(f"Weighted F1:                    {test_results['weighted_f1']:.4f}")
print(f"Top-2 Accuracy:                 {test_results['top2_accuracy']:.4f}")
print(f"Top-3 Accuracy:                 {test_results['top3_accuracy']:.4f}")

print("\nSaved ZIP:")
print(ZIP_PATH)

FINAL MODELING SUMMARY
Input mapped rows:              73484
Rows after 7-domain mapping:    72431
Number of features:             54
Number of domains:              7
Training rows:                  46355
Validation rows:                11589
Test rows:                      14487
Best CatBoost iteration:        528

Final test metrics:
Accuracy:                       0.4465
Balanced Accuracy:              0.3862
Macro F1:                       0.3871
Weighted F1:                    0.4430
Top-2 Accuracy:                 0.6731
Top-3 Accuracy:                 0.8071

Saved ZIP:
/content/riasec_final_model_files.zip
